In [1]:
__author__ = 'Vinesh Maguire Rajpaul'
__email__ = 'vr325@cantab.ac.uk'

# Imports

In [2]:
# Packages from the Python Standard Library
import copy
import pickle
import os
import sys
from collections import Counter

# Packages for numerical and data analysis
import numpy as np
import pandas as pd
from scipy.stats import triang as triang

# Packages for plotting/data visualisation
import matplotlib.pyplot as plt
from matplotlib import ticker
from matplotlib import lines, patches
from matplotlib.colors import to_rgb, to_hex

# Specify data files to be used in analysis

In [4]:
# ONS data: APS; total workforces (all employee types, full- + part-time, etc.)
APS_FILE = '../data/raw/people_in_jobs/APS-SOC20-2024.xlsx'

# ONS data: JOBS04; self-employed workforces in UK and four nations
JOBS04_FILE = '../data/raw/self-employment/jobs04mar2025.xls'

# ONS data: statistical mapping between SOC and SIC codes in UK workforce
SOC_SIC_FILE = ('../data/raw/SOC-SIC/' + 
                '4digitoccupationbyvariousfactorsj23j24.xlsx')

# ONS data: UK and national GVA disaggregated by SIC-2d codes
GVA_SIC2d_FILE = (
    '../data/raw/GVA/regionalgrossvalueaddedbalanced' + 
    'byindustryandallinternationalterritoriallevelsitlregions.xlsx')

# ONS data: time series giving UK's total GVA per annum and per quarter
TOTAL_GVA_FILE = '../data/raw/GVA/GVA-series-170325.csv'

# AcadMathSci data: table for mapping between different SIC encodings in ONS 
# datasets (GVA datasets, ASHE tables, APS data, etc. all use different
# levels of SIC granularity and/or formatting, e.g. "09" instead of "9", even 
# though all are from the ONS)
SIC_TRANSLATION_FILE = '../data/SIC_translation.xlsx'

# AcadMathSci data: estimated MS fracs vs SOC codes
MS_FRAC_FILE = '../data/MSfracs_2025_10.xlsx'

# Define various helper functions

## General-purpose functions

In [5]:
def flatten(xss):
    """
    Flatten a list of lists into a single list.
    """
    return [x for xs in xss for x in xs]

def my_round(x, base=5):
    """
    Round a number to specified base. E.g. my_round(73) = 75,
    my_round(72) = 70, when using base 5 in both cases.
    """
    return base * round(x/base)

def renorm_matrix(A, axis=0):
    """
    Renormalise a matrix by rows (axis 0) or columns (axis 1) so that
    entries along specified axis sum to 1. I.e. return a right (axis 0)
    or left (axis 1) stochastic matrix.
    """
    if axis == 1: A = A.T

    for row_ix in range(A.shape[0]):
        row_sum = np.sum(A[row_ix,:])
        if row_sum > 0:
            A[row_ix, :] /= row_sum

    return A if axis==0 else A.T

## Functions particular to SOC-SIC-GVA analysis

In [6]:
def DF_to_vec(df, lookup_col, vec_keys, fill_value = 0):
    """
    Map items in DataFrame column lookup_col to an output vector, where the 
    vec_keys. Useful if, e.g., DataFrame contains only non-zero/non-trivial
    records but must be broadcast to a larger vector including zero/trivial
    values, as may be needed for linear algebraic operations in this analysis.
    """
    vec_out = fill_value*np.ones(len(vec_keys))
    remapped_counter = 0
    for ix, key in enumerate(vec_keys):
        try:
            vec_out[ix] = df.loc[key,lookup_col]
            remapped_counter += 1
        except KeyError:
            pass
    if remapped_counter ==0:
        warn('No cells from input DataFrame were mapped to the output vector')
    return vec_out

def compute_SOC_SIC_map():
    """
    Function to load ONS SOC-SIC (statistical) mapping, pre-process
    it (e.g. aggregating 4d codes to 2d), and output (i) tuples of all
    SOC codes, SIC-2d codes, SIC-4d codes, and (ii) matrices (arrays)
    that map SOC codes to SIC-2d or SIC-4d codes. This is important in
    the economic contribution analysis because we'll need to transform 
    workforce vectors (MS, non-MS, whole UK) from an SOC (occupation) 
    basis to a SIC (industry) basis.
    """
    
    # NB: check if hard-coded sheet index, footer rows etc. are 
    # still correct if re-doing with data for a year after 2024
    SS_df = pd.read_excel(SOC_SIC_FILE,
                 header=8,sheet_name=3,index_col=1, skipfooter=12)
                

    # Extact (4-digit) SOC codes from DataFrame index
    SOC_codes = [SOC[0:4] for SOC in SS_df.index]

    # Replace all text entries with zeros (via NaNs) -
    # correspond to occupation/industry pairs with insufficient data
    for key in SS_df.keys():
        SS_df[key] = pd.to_numeric(SS_df[key],errors='coerce').values

    # Transpose DataFrame so SIC codes label rows
    SS_df=SS_df.T
    SS_df.drop('Unnamed: 0', inplace=True)

    # --- Convert DataFrame Index to  MultiIndex and thus get SIC codes ---
    SIC_codes = []

    for key in SS_df.index:
        #print(key)
        start = key.index(' ') + 1
        stop2d, stop4d = start + 2, start + 5
        SIC_codes.append((key[start:stop2d],key[start:stop4d]))

    SS_df.iloc[np.isnan(SS_df).values]=0 # replace NaNs with zeros

    SS_df = SS_df.set_index(pd.MultiIndex.from_tuples(SIC_codes))
    SS_df.drop('99', inplace=True) # SIC code 99 not counted in GVA

    # (Syntactically easier now to work with MultiIndex over rows)
    SS_df = SS_df.T

    # Get list of SIC-2d and SIC-4d codes from MultiIndex
    SIC2d_codes = sorted(set(SS_df.columns.get_level_values(0)))
    SIC4d_codes = sorted(set(SS_df.columns.get_level_values(1)))

    # Populate SOC-SIC matrix (marginalising SIC4d to SIC2d where needed)
    SOC_to_SIC4d = SS_df.values
    SOC_to_SIC2d = np.zeros([len(SOC_codes),len(SIC2d_codes)])
    for col, s2d in enumerate(SIC2d_codes):
        SOC_to_SIC2d[:,col] = SS_df[s2d].sum(axis=1).values

    # Normalise matrices
    SOC_to_SIC2d = renorm_matrix(SOC_to_SIC2d)
    SOC_to_SIC4d = renorm_matrix(SOC_to_SIC4d)

    # The list of SOC and SIC codes are immutable, so return them as tuples
    return (tuple(SOC_codes), tuple(SIC2d_codes), tuple(SIC4d_codes),
        SOC_to_SIC2d, SOC_to_SIC4d)


In [7]:
SOC_codes, SIC2d_codes, SIC4d_codes, SS2d, SS4d = compute_SOC_SIC_map()

# Load and pre-process raw (ONS) data

In [8]:
# Display up to 200 rows of Pandas DataFrame
pd.set_option('display.max_rows', 200)

## Load GVA vs industry data for UK & four nations

In [9]:
# Load GVA-vs-SIC2d data for whole UK and store in Pandas DataFrame
GVA = pd.read_excel(GVA_SIC2d_FILE, sheet_name='Table 1c',skiprows=1, dtype={'SIC07 code': str})
GVA = GVA.loc[GVA['Region name']=='United Kingdom'].reset_index() # select UK instead of a nation
GVA = GVA.set_index('SIC07 code')
GVA.loc['36-37']=GVA.loc['36']+GVA.loc['37'] # merge SIC-2d codes 36 and 37, 

# Load UK total GVA time series
UK_GVA = pd.read_csv(TOTAL_GVA_FILE, header=None)
GVA_years = [str(yr) for yr in np.arange(2010,int(2024)+1)]
UK_GVA = ({year: 0.001*np.double(UK_GVA[1][np.where(
    UK_GVA[0].values == year)[0][0]]) for year in GVA_years})

In [10]:
# Repeat the above, but now for each of the UK's four nations

nation_list = ['United Kingdom', 'England', 'Wales', 'Scotland', 'Northern Ireland']
GVA_nation = {}
for nation in nation_list:
    GVA_temp = pd.read_excel(GVA_SIC2d_FILE, sheet_name='Table 1c',skiprows=1, dtype={'SIC07 code': str})
    GVA_temp = GVA_temp.loc[GVA_temp['Region name']==nation].reset_index()
    GVA_temp = GVA_temp.set_index('SIC07 code')
    GVA_temp.loc['36-37'] = GVA_temp.loc['36']+GVA_temp.loc['37']
    GVA_nation[nation] = GVA_temp

In [11]:
# Load table for translating between SIC codes in different ONS datasets,
# and store as Pandas DataFrame

T = pd.read_excel(SIC_TRANSLATION_FILE,dtype=str)
T = T.set_index('SIC2d')

# Number of SIC-2d codes in translation file should be identical to
# number of SIC-2d codes in SOC-SIC mapping computed above
assert list(SIC2d_codes)==list(T.index)

## Load disaggregated DE/SE workforce data in UK & four nations

In [12]:
# Shorthand: DE = directly-employed; SE = self-employed

def get_DE_SE_jobs(ASHE_key):
    """
    Load DE and SE workforces for files labelled with ASHE_key (e.g., the key 
    "2011p" identifies provisional 2011 ASHE data, whereas "2023r" identifies 
    revised 2023 data, per ONS shorthand) and save the workforce in a 
    Pandas DataFrame, WF, for use in subsequent analysis.
    """
    
    # Extract jobs year directly from ASHE_key
    output_year = ASHE_key[0:-1]

    T = pd.read_excel(SIC_TRANSLATION_FILE, dtype=str)

    # --- Get directly employed (DE) workers: ASHE Table 5.5a ---

    WF_keys = list(set(T['GVA_SIC2d'].values))
    WF = {key:{'UK_DE':0, 'UK_SE':0, 'MSW': 0,
               'MS_GVA': 0} for key in WF_keys}

    # Locate directory corresponding to relevant ASHE key
    ASHE5_dir = f'../data/raw/ASHE_Table5/{ASHE_key}/'

    # Read all data in ASHE Table 5 directory. Skip initial footer rows,
    # specify input data types for some columns, drop irrelevant columns, etc.
    # NB: check if hard-coded sheet index, footer rows etc. are 
    # still correct if re-doing with data for a year after 2024
    ASHE5_files = os.listdir(ASHE5_dir)
    ASHE5 = pd.read_excel(ASHE5_dir + ASHE5_files[[
        'Table 5.5a' in filename for filename in ASHE5_files].index(True)], 
                          sheet_name='All', header=4, 
                          usecols='A:C', skipfooter=6)
    ASHE5['Code'] = ASHE5['Code'].astype(str)
    ASHE5['DE_jobs'] = ASHE5['(thousand)']
    ASHE5.drop('(thousand)', axis=1, inplace=True)
    ASHE5 = ASHE5.iloc[np.arange(20,128)] 

    ASHE5_counted = []
    for key, val in WF.items():
        #print(key, T['ASHE4or5_SIC2d'].loc[T['GVA_SIC2d']==key].values)

        # manually remove DE jobs in code 09 from total for SIC-letter B to 
        # compute total DE jobs in SIC-2d codes 05--08.
        if key == '5-8':
            DE_jobs = np.sum(
                ASHE5.loc[ASHE5['Code'].astype(str)=='B','DE_jobs'].values-
                ASHE5.loc[ASHE5['Code'].astype(str)=='9','DE_jobs'].values)
            WF[key]['UK_DE'] += DE_jobs
            ASHE5_counted.append('B')
            
        # for codes 11--12, use only code 11 (no DE job data available for 
        # code 12; fortunately, it represents negligible % oftotal workforce)
        elif key == '11-12':
            DE_jobs = np.sum(
                ASHE5.loc[ASHE5['Code'].astype(str)=='11','DE_jobs'])
            WF[key]['UK_DE'] += DE_jobs
            ASHE5_counted.append('11')
            ASHE5_counted.append('12')
            
        # for all other codes, processing of DE jobs is straightforward
        else:
            ASHE_codes = set(
                T.loc[T['GVA_SIC2d']==key,'ASHE4or5_SIC2d'].values)
            for code in ASHE_codes:
                DE_jobs = ASHE5.loc[ASHE5['Code']==str(code),'DE_jobs'].sum()
                if isinstance(DE_jobs,str):
                    DE_jobs = 0
                elif code not in ASHE5_counted:
                        WF[key]['UK_DE'] += DE_jobs
                        ASHE5_counted.append(code)

    # --- Get self-employed (SE) workers from JOBS04 table ---

    # NB: check if hard-coded sheet index, footer rows etc. are 
    # still correct if re-doing with data for a year after 2024
    JOBS04 = pd.read_excel(JOBS04_FILE,
                         sheet_name='1. UK totals', header=5, skipfooter=10)
    years=([str(quarter).replace('(r)','').replace('(p)','').strip()[-2::]
            for quarter in JOBS04['SIC 2007 division'].values])
    JOBS04['years']= years
    JOBS04 = JOBS04.drop(JOBS04.columns[[0,1,-2,-3]],axis='columns')
    JOBS04 = JOBS04.loc[JOBS04['years']==output_year[-2::]]
    JOBS04 = (JOBS04.drop('years',axis='columns')).mean()
    JOBS04.index = JOBS04.index.astype('str')

    SE_tot, JOBS04_counted = 0, []

    for key, val in WF.items():
        # Translate SIC2d code format from GVA data to format in JOBS04 data
        JOBS04_codes = set(T.loc[T['GVA_SIC2d']==key,'JOBS0x_SIC2d'].values)

        for code in JOBS04_codes:
            # Locate the SE jobs listed under that SIC2d code
            SE_jobs = JOBS04[code]
            
            if code not in JOBS04_counted:
                # If not already counted, add to workforce vector
                # NB: latter vector based using GVA_SIC2d, not JOBS04_SIC2d
                # (slightly different granularity, etc.) hence need for
                # manual checking, avoiding double-counting, etc.
                WF[key]['UK_SE'] += SE_jobs
                SE_tot += SE_jobs
                JOBS04_counted.append(code)

    for g in WF.values():
        # Add together DE and SE jobs to get total UK workers (UKW)
        g['UKW'] = g['UK_SE'] + g['UK_DE']

    return WF

# Get workforce (WF) DataFrame for specified year and 
# provisional/revised dataset
WF_ASHE_JOBS04 = get_DE_SE_jobs('2024p')

## Load APS data: total workforces across UK & four nations

In [42]:
# NB: check if hard-coded sheet index, footer rows etc. are 
# still correct if re-doing with data for a year after 2024
WF = pd.read_excel(APS_FILE,
                  sheet_name=0, header=[11,12], index_col=0,skipfooter=4)

# Extract SOC2020 (4 digit) codes and descriptions from DataFrame index
SOC20descr_all = [str(i) for i in WF.index]
SOC20_all = [str(i)[0:4] for i in WF.index]

# Get rid of first element in list (column sub-headers)
SOC20descr_all.pop(0)
SOC20_all.pop(0)

# List of nations (including UK) formatted per this DataFrame's columns
all_nations = list(WF.columns.levels[0])

# Make dictionary to store various statistics about workforces in
# UK and four nations (e.g. total jobs, no. of jobs that had to be
# estimated because of missing data, etc.)
national_info = {nation: dict(total=WF.loc['Total',(nation,'number')],
                           total_conf=WF.loc['Total',(nation,'conf')],
                 n_jobs_est=0,  jobs_est=0, n_conf_estimated=0)
                 for nation in all_nations}

# Drop "Total" as it's not a SOC code
WF.drop('Total', inplace=True)
# After removing "Total", should be same no. of SOC codes as in SOC20_all
assert(len(SOC20_all)==len(WF)) 

# Define a column that contains only SOC2020 code but not description
WF['SOC20'] = SOC20_all
# Define a column that will store MS fraction of each SOC2020 occupation
WF['MS_frac'] = np.zeros(len(SOC20_all))
# Index the DataFrame by SOC2020 (4-digit) code
WF = WF.set_index(WF['SOC20'].values)

## Load MS fractions (AcadMathSci estimates)

In [47]:
# AcadMathSci (2025 analysis using mostly 2024 data)
MS_fracs_AMS = pd.read_excel(MS_FRAC_FILE,
                  sheet_name=0, header=0, index_col=0)
MS_fracs_AMS.index = MS_fracs_AMS.index.map(str)

# Ensure all MS fractions in WF DataFrame are zero (default) and define
# two other columns (max and min fraction) of equal dimension
WF['MS_frac'] *= 0
WF['MS_frac_min'] = 0*WF['MS_frac']
WF['MS_frac_max'] = 0*WF['MS_frac']

# Get AcadMathSci MS fractions (best estimate, min, max)
for ix, soc in enumerate(MS_fracs_AMS.index):
    
    WF.loc[soc, 'MS_frac'] = MS_fracs_AMS.loc[soc]['MS_frac']
    WF.loc[soc, 'MS_frac_min'] = MS_fracs_AMS.loc[soc]['MS_frac_min']
    WF.loc[soc, 'MS_frac_max'] = MS_fracs_AMS.loc[soc]['MS_frac_max']

## Pre-process national workforce data

In [48]:
# Estimate missing job numbers, missing uncertainty estimates, etc. 
for nation in all_nations:

    # Fraction: nation's population vs UK population (= 1 for UK)
    nation_frac = (national_info[nation]['total']/
                   national_info['United Kingdom']['total'])

    for ix in WF.index:

        # get total jobs and 95% CIs
        jobs = str(WF.loc[ix,(nation,'number')])
        conf = str(WF.loc[ix,(nation,'conf')])
        
        # string entry for jobs indicates missing/insufficient data
        if not jobs.isdigit():
            
            # estimate national jobs based on national population fraction and
            # UK-wide total for relevant occupation. Note: only needed
            # for very small fraction of national workforces
            jobs_est = nation_frac*WF.loc[ix,('United Kingdom','number')]
            WF.loc[ix,(nation,'number')] = jobs_est
            # Count number of occupations where jobs had to be estimated
            national_info[nation]['n_jobs_est'] += 1
            national_info[nation]['jobs_est'] += jobs_est

        # ditto for confidence interval, if string
        if not conf.isdigit():
            # 95% CIs, worst case model per ONS: 46*sqrt(N); median 45*sqrt(N)
            conf_estimated = 45* np.sqrt(WF.loc[ix,(nation,'number')])/2 
            WF.loc[ix,(nation,'conf')] = conf_estimated
            # Count number of occupations where CIs had to be estimated
            national_info[nation]['n_conf_estimated'] += 1

    # Re-scale +/- 2-sigma (95%) to +/- 1-sigma (68%) CIs
    WF[(nation, 'conf')] /= 2

# Pickle workforce dataframe and write to disk
with open('../data/processed/MSfracs_SOC20_2025.pickle', 'wb') as handle:
    pickle.dump(WF, handle, protocol=pickle.HIGHEST_PROTOCOL)

## Helper functions: random realisations of workforces

In [49]:
def parametrize_triang_pdf(a, mode, b):
    """
    Given a triangular probabily density function (PDF) with  lower limit = a, 
    mode = mode, and upper limit = b, return the scale and shape parameters 
    needed by scipy.stats's triang()
    """

    assert (a <= mode <= b)
    
    loc = a
    scale = b - a
    shape = (mode - a) / scale
    
    return shape, loc, scale
    
def generate_MSW_pdfs():
    """
    Extract SOC codes from WF (workforce DataFrame) and then generate 
    triangular PDFs based on AcadMathSci lower, modal, and upper estimates of 
    each SOC code's MS fraction
    """

    # Dict to map each SOC code to a PDF
    MSW_pdfs = {}

    # Loop over all SOC codes in workforce
    for soc in WF.index:

        # Extract triangular PDF limits and mode
        mode = WF.loc[soc,'MS_frac'].iloc[0]
        a = WF.loc[soc,'MS_frac_min'].iloc[0]
        b = WF.loc[soc,'MS_frac_max'].iloc[0]
        
        if (a == mode == b):
            # If point-estimate only (e.g. MS frac = 100%)
            pdf = mode
        else:
            # Else generate triangular PDF via scipy.stats
            shape, loc, scale = parametrize_triang_pdf(a, mode, b)
            pdf = triang(shape, loc, scale)

        # Store that SOC code's PDF
        MSW_pdfs[soc] = pdf
        
    return MSW_pdfs  
    
def draw_random_WF(WF, nation, 
                   randomise_overall=True, randomise_MSW=True):
    """
    Given a workforce DataFrame WF, generate a random realisation of a nation's
    workforce based on (i) the (Gaussian) 1-sigma confidence intervals on 
    total workers in each occupation, per ONS data, and/or (ii) on
    the triangular PDFs, per AcadMathSci estimates, that model uncertainty in 
    the MS fractions within given occupations. I.e., generate random
    realisations that account for ONS-level sampling uncertainty and/or
    AcadMathSci's uncertainty in occupations' MS fractions.
    """

    # Make deep copy to ensure we don't accidentally modify the original WF
    wf = copy.deepcopy(WF[nation])
    
    for soc in WF.index:

        # UKW = UK workers; shorthand for all workers in an occupation, as 
        # opposed to MSW = MS workers. But I re-use this short-hand even
        # if it's e.g. Wales: UKW = total workers in given occupation in Wales.
        
        UKW_rand = WF.loc[soc][nation]['number']

        if randomise_overall:
            UKW_conf = WF.loc[soc][nation]['conf']
            UKW_rand += np.random.randn()*UKW_conf
            
        if randomise_MSW:
            if isinstance(MSW_pdfs[soc], (int, float)):
                MS_frac_rand = MSW_pdfs[soc]
            else:
                MS_frac_rand = MSW_pdfs[soc].rvs()
        else:
            MS_frac_rand = WF.loc[soc,'MS_frac'].iloc[0]

        # Store random realisations of MS fractions, total workers, and MS
        # workers (all vectors with no. elements = no. of SOC2020 codes)
        wf.loc[soc,'MS_frac_rand'] = MS_frac_rand
        wf.loc[soc, 'UKW_rand'] = UKW_rand
        wf.loc[soc,'MSW_rand'] = MS_frac_rand*UKW_rand

    return wf


In [50]:
MSW_pdfs = generate_MSW_pdfs()

# Optimise SOC-SIC mapping (whole UK only)

Some detailed notes are given below to illuminate the need for and approach to optimising the SOC-SIC mapping (refer also to AcadMathSci's detailed methodology paper for more context).
* ``compute_SOC_SIC_map()`` extracts an initial SOC-SIC mapping (matrix), ``SS2d``, from ONS data that constrain the statistical relationship between 4-digit SOC codes (occupations) and 4-digit SIC codes (industries) in the UK workforce. 
* The relevant data file used (as of September 2025) is ``4digitoccupationbyvariousfactorsj23j24.xlsx``.
    - But these data cover only full-time, directly-employed workers (some 25 mn workers). By contrast, our analysis deals with the entire UK workforce (some 33 mn workers), including part-time and self-employed workers.
    - These data also make no distinction between MS (mathematical science) and non-MS workers.
* Thus, ``SS2d`` should be a reasonable first approximation (perhaps to within about 10-15%) of the occupation-to-industry mapping for MS workers specifically, but it won't be exactly correct.
* Indeed, when using ``SS2d`` at face value to re-base the UK workforce from occupations to industries, we end up with a workforce vector with some industries containing more workers than the total workforce in those industries, per separate ONS datasets (e.g., JOBS04 + ASHE datasets).
* Therefore we use ``SS2d`` as a starting point in an optimisation algorithm, below, to refine the mapping.
* In abstract terms, we want to solve $\vec{x}_\text{SOC}\cdot\mathbb{S}=\vec{x}_\text{SIC}$ for $\mathbb{S}$; there are infinitely many matrices that will satisfy this equation, so the problem is degenerate. But we already have a reasonable estimate or starting guess for $\mathbb{S}$, ${\mathbb{S}}_0$ (i.e., ``SS2d``), so an approximate iterative solution is possible, e.g., via gradient descent $\mathbb{S}_{i+1} \leftarrow {\mathbb{S}_i} - \eta\; \vec{x}_\text{SOC}^\text{T} \cdot\left(\vec{x}_\text{SOC}\cdot\mathbb{S} - \vec{x}_\text{SIC} \right)$, repeated until convergence, where $\eta>0$ is a step size parameter. This is just the matrix version of the Widrow–Hoff rule/Least Mean Squares algorithm; intuitively, the solution is refined by moving in the direction that leads to the biggest reduction in error.
* The code below optimises $\mathbb{S}$ semi-heuristically, more or less via gradient descent, but
    - adjusts $\mathbb{S}$ only for occupations/industries relevant to the MS workforce analysis;
    - takes into account slightly different SIC granularity in different ONS datsets;
    - re-normalises $\mathbb{S}$ every iteration to ensure $||\vec{x}_\text{SOC}||_1. =||\vec{x}_\text{SIC}||_1$ (i.e., total workforce size can never change); and
    - is constrained to adjust $\mathbb{S}$ one industry at a time, rather than descending in an arbitrary direction in occupation-industry space (which creates challenges with re-normalisation and numerical stability).

In [51]:
# Algorithm is numerically stable, fast, and achieves convergence to a 
# fixed point or (very small limit cycle) after suitably large no. of 
# iterations; little difference in results between, e.g., 200 and 500 
# iterations. Note: the seemingly large number of iterations to convergence
# stems from the constraint to adjust only one industry per iteration.

N_SOC_SIC_OPTIMISE = 400

SOC_codes, SIC2d_codes, SIC4d_codes, SS2d, SS4d = compute_SOC_SIC_map()

# Make a copy of SS2d; optimise the copy, not the original
SS2d_opt = copy.deepcopy(SS2d)

# Extract MS worker and total worker vectors for whole UK (SOC basis)

wf = draw_random_WF(WF, 'United Kingdom', 
                    randomise_overall=False, randomise_MSW=False)
MSW_SOC20_vec = wf['MSW_rand'].values
UKW_SOC20_vec = wf['UKW_rand'].values

print(f'\nSOC-SIC adjustment iteration: ', end='', flush=True)

for j in range(N_SOC_SIC_OPTIMISE):

    # Transform workforce vectors from SOC to SIC basis via SS2d matrix
    MSW_SIC07_vec = MSW_SOC20_vec@SS2d_opt
    UKW_SIC07_vec = UKW_SOC20_vec@SS2d_opt
                                        
    # Populate DataFrame: UK and MS workers (SIC basis)
    for ix, SIC2d_code in enumerate(T.index):
        T.loc[SIC2d_code,'UKW'] = UKW_SIC07_vec[ix]
        T.loc[SIC2d_code,'MSW'] = MSW_SIC07_vec[ix]

    # Create pivot table to facilitate easy manipulation of GVA data
    # vs SIC-2d codes (e.g. marginalising GVA over SIC-2d codes)
    GVA_pivot = pd.pivot_table(T, values=['UKW','MSW'], columns=['GVA_SIC2d'], 
                               aggfunc='sum').transpose()
    
    if j%5 == 0 : print(j, end='|', flush=True)

    # (i) = expected SE+DE workers vs SIC-2d codes from JOBS04. 
    # Note: WF_ASHE_JOBS04 already indexed by GVA_SIC2d index, so 
    # no SIC code translation needed here. 
    
    GVA_pivot['UKW_SE_DE'] = 0*GVA_pivot['UKW'] # Avoids creating pointer
    for key in WF_ASHE_JOBS04.keys():
        GVA_pivot.loc[key,'UKW_SE_DE'] += (WF_ASHE_JOBS04[key]['UKW']*1000)

    # (ii) = ratio of mapped total workers (in each SIC code) vs 
    # total workers (SE + DE) inferred from JOBS04 data
    GVA_pivot['UKW_ratio'] = GVA_pivot['UKW']/GVA_pivot['UKW_SE_DE']

    # (iii) = abs. divergence between expected and observed ratio (ii)
    GVA_pivot['UKW_ratio_dev'] = np.abs(1-GVA_pivot['UKW_ratio'])

    # (iv) = number of MS workers in a SIC code weighted by the divergence
    # (iii); quantifies extent to which SOC-SIC mapping is sub-optimal 
    # for MS workers (i.e.: no need to focus on optimising the mapping for 
    # occupations/industries irrelevant to subsequent MS-focused analysis).
    GVA_pivot['weighted_MS_dev'] = (
        GVA_pivot['UKW']*GVA_pivot['UKW_ratio_dev'])

    # Get SIC-2d codes corresponding to maximal workforce mismatch
    GVASIC2d_max_mismatch = GVA_pivot.sort_values(
        'weighted_MS_dev', ascending=False).iloc[0].name
        
    SIC2d_codes_match = ([i for i in 
                          T.loc[T['GVA_SIC2d']==GVASIC2d_max_mismatch].index])
    
    # Then find the index/indices of those SIC-2d codes
    SIC2d_ixs_match = [SIC2d_codes.index(k) for k in SIC2d_codes_match]

    # Compute factor by which SS2d column(s) should be adjusted
    adjustment_factor = GVA_pivot.loc[GVASIC2d_max_mismatch]['UKW_ratio']

    # Adjust maximal mismatch column(s) and renormalise so total workforce
    # size always remains unchanged
    SS2d_opt[:,SIC2d_ixs_match]/= adjustment_factor
    SS2d_opt = renorm_matrix(SS2d_opt)


SOC-SIC adjustment iteration: 0|5|10|15|20|25|30|35|40|45|50|55|60|65|70|75|80|85|90|95|100|105|110|115|120|125|130|135|140|145|150|155|160|165|170|175|180|185|190|195|200|205|210|215|220|225|230|235|240|245|250|255|260|265|270|275|280|285|290|295|300|305|310|315|320|325|330|335|340|345|350|355|360|365|370|375|380|385|390|395|

The optimised SOC-SIC mapping, derived using all-UK data as a starting point, is now re-used for GVA calculations in all four nations. (As a reminder, workforces have to be re-based from occupations to industries, i.e. SOC to SIC codes, as ONS publishes GVA data disaggregated by industry but not occupation). 

The signal-to-noise ratios would have been too low to derive separate SOC-SIC mappings for each of the four nations, except perhaps England, which anyway constitutes around 85% of the UK's population.

# Monte-Carlo analysis: UK and four nations

To quantify the uncertainty in various quantities we estimate - particularly the total sizes of the non-MS and MS workforces in each nation, and their associated GVA contributions and economic productivities - we now run a series of Monte Carlo simulations, allowing for variation within both sampling uncertainty (e.g. in ONS datasets describing the whole UK workforce) and uncertainty in MS fractions across different occupations.

In [53]:
# Scale factor to transform SIC-disaggregated GVA data for 2023 to 2024 
# using ratio of (UK total GVA in 2024)/(UK total GVA in 2023). Note: for 2024,
# SIC-disaggregated GVA data were not available at time of analysis.

GVA_rescaling = UK_GVA['2024']/UK_GVA['2023']

# The list of nations extracted from APS tables contains 'England and Wales' 
# and 'Great Britain', neither of which is needed here; so, specify manually.

nation_list = ['United Kingdom', 'England', 'Wales', 
               'Scotland', 'Northern Ireland']

In [ ]:
# Exact number of Monte Carlo trials is obviously not critical; all else being
# equal, more trials lead to more refined uncertainty estimates, though
# computation time is quite high for these trials so returns diminish quickly.

N_RAND_TRIALS = 200

# Seed random number generators (for reproducibility)
np.random.seed(0)

# Dictionary to store Monte Carlo results for each nation
nation_rand_results = {}

for nation in nation_list:
    
    print('\n\n' + '*'*50 + '\n' + nation + '\n' + '*'*50)

    # Random results for current nation
    rand_results = []
    
    for i in range(N_RAND_TRIALS):
         
        print(f'{i}|', end='', flush=True)
    
        # Draw random realisation of workforce
        wf = draw_random_WF(WF, nation, randomise_overall=True,
                           randomise_MSW=True)

        # Extract MS worker and total worker vectors
        MSW_SOC20_vec = wf['MSW_rand'].values
        UKW_SOC20_vec = wf['UKW_rand'].values
         
        # Transform vectors from SOC to SIC basis via SS2d matrix
        MSW_SIC07_vec = MSW_SOC20_vec@SS2d_opt
        UKW_SIC07_vec = UKW_SOC20_vec@SS2d_opt
                                            
        # Populate DataFrame: UK and MS workers (basis: SIC not SOC)
        for ix, SIC2d_code in enumerate(T.index):
            T.loc[SIC2d_code,'UKW'] = UKW_SIC07_vec[ix]
            T.loc[SIC2d_code,'MSW'] = MSW_SIC07_vec[ix]

        # Create pivot table to facilitate easy manipulation of GVA data
        # vs SIC-2d codes (e.g. marginalising GVA over SIC-2d codes)
        GVA_pivot = pd.pivot_table(
            T, values=['UKW','MSW'], columns=['GVA_SIC2d'],
            aggfunc='sum').transpose()
        
        # Re-scale all GVAs from 2023 to 2024 terms (using UK-wide data)
        for GVA_SIC2d_code in GVA_pivot.index:
            GVA_pivot.loc[GVA_SIC2d_code,'GVA'] = (
                GVA_nation[nation].loc[
                GVA_SIC2d_code,'2023']*GVA_rescaling/1000) # Unit conv.

        # Compute MS GVA via sum((MS fraction * total workforce) * GVA)
        GVA_pivot['MS_GVA'] = (
            GVA_pivot['MSW']/GVA_pivot['UKW']*GVA_pivot['GVA'])
        total_MS_GVA = GVA_pivot['MS_GVA'].sum()
 
        # Add to list of dicts: result for i'th random/Monte Carlo iteration
        rand_results.append(dict(
            UKW_SOC20_vec = UKW_SOC20_vec, MSW_SOC20_vec = MSW_SOC20_vec,
            UKW_SIC07_vec = UKW_SIC07_vec, MSW_SIC07_vec = MSW_SIC07_vec,
            MSW_GVA = total_MS_GVA, 
            GVA_pivot=GVA_pivot, SS2d=SS2d_opt))
            
    # Save the Monte Carlo results for the current nation
    nation_rand_results[nation] = rand_results



**************************************************
United Kingdom
**************************************************
0|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|26|27|28|29|30|31|32|33|34|35|36|37|38|39|40|41|42|43|44|45|46|47|48|49|50|51|52|53|54|55|56|57|58|59|60|61|62|63|64|65|66|67|68|69|70|71|72|73|74|75|76|77|78|79|80|81|82|83|84|85|86|87|88|89|90|91|92|93|94|95|96|97|98|99|100|101|102|103|104|105|106|107|108|109|110|111|112|113|114|115|116|117|118|119|120|121|122|123|124|125|126|127|128|129|130|131|132|133|134|135|136|137|138|139|140|141|142|143|144|145|146|147|148|149|150|151|152|153|154|155|156|157|158|159|160|161|162|163|164|165|166|167|168|169|170|171|172|173|174|175|176|177|178|179|180|181|182|183|184|185|186|187|188|189|190|191|192|193|194|195|196|197|198|199|

**************************************************
England
**************************************************
0|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|26|27|28|29

# Summarise results for UK & four nations

In [ ]:
# For storing final estimates, not full Monte Carlo simulation results
nation_results = {}

for nation in nation_list:

    print('\n\n' + '*'*50 + '\n' + nation + '\n' + '*'*50)
    
    # Extract workforces without any random variation (i.e., as was required
    # previously for Monte Carlo simulations)
    wf_norand = draw_random_WF(WF, nation, randomise_overall=False,
                   randomise_MSW=False)

    #  Extract vectors corresponding to MS workers, UK workers
    MSW_SOC20_vec_norand = wf_norand['MSW_rand'].values, 
    UKW_SOC20_vec_norand = wf_norand['UKW_rand'].values

    # Convert MS & total workers from SOC to SIC basis via SS2d matrix
    MSW_SIC07_vec_norand = MSW_SOC20_vec_norand@SS2d_opt
    UKW_SIC07_vec_norand = UKW_SOC20_vec_norand@SS2d_opt

    # Store the above workforce vectors in a dictionary
    vectors = dict(MSW_SOC20_vec_norand=MSW_SOC20_vec_norand,
                   UKW_SOC20_vec_norand=UKW_SOC20_vec_norand,
                   MSW_SIC07_vec_norand=MSW_SIC07_vec_norand,
                   UKW_SIC07_vec_norand=UKW_SIC07_vec_norand 
                  )

    # For each nation, print out mean and std (from Monte Carlo trials) of:
    
    # ...total MS GVA
    rand_GVA = np.array([y['MSW_GVA'] for y in nation_rand_results[nation]]) 
    print(
        f'MS direct GVA:\t\t£{np.mean(rand_GVA):.2f} bn ± £{np.std(rand_GVA):.2f} bn')

    # ...total MS workers
    rand_MSW = np.array([sum(y['MSW_SIC07_vec']) 
                         for y in nation_rand_results[nation]])
    print(f'MS workforce size:\t{np.mean(rand_MSW)/1e6:.3f} mn ' + 
          f'± {np.std(rand_MSW)/1e6:.3f} mn')

    # ...total UK workers (MS + non-MS workers)
    rand_UKW = np.array([sum(y['UKW_SIC07_vec']) 
                         for y in nation_rand_results[nation]])
    print(f'Total workforce size:\t{np.mean(rand_UKW)/1e6:.3f} mn ' + 
          f'± {np.std(rand_MSW)/1e6:.3f} mn')


    # ...MS worker productivity (GVA per job)
    rand_prod = np.array(([y['MSW_GVA']*1e6/sum(y['MSW_SIC07_vec']) 
                           for y in nation_rand_results[nation]])) 
    print('MS worker prod.:\t' + f'£{1e3*np.mean(rand_prod):.0f}/job' + 
          f' ± £{1e3*np.std(rand_prod):.0f}/job')

    # ...non-MS worker productivity
    # (GVA_nation, from ONS data, not yet rescaled to 2024)
    nation_GVA = GVA_rescaling*(
        GVA_nation[nation]['2023']['Total']/1e3) 
    rand_nonMSprod = (
        [(nation_GVA-rand_GVA)/
         (rand_UKW-rand_MSW)*1e6 for 
         y in nation_rand_results[nation]]); 
    print('Non-MS worker prod.:\t' + f'£{1e3*np.mean(rand_nonMSprod):.0f}/job' + 
          f' ± £{1e3*np.std(rand_nonMSprod):.0f}/job', end='')

    # Store all of the above in a dictionary corresponding to current nation
    nation_results[nation] = dict(
        nation_GVA = nation_GVA, vectors=vectors, 
        rand_GVA=rand_GVA, rand_MSW=rand_MSW, rand_UKW=rand_UKW,
        rand_prod=rand_prod, rand_nonMSprod=rand_nonMSprod,
        wf_norand=wf_norand, 
        MSW_SIC07_vec_norand=MSW_SIC07_vec_norand, 
        MSW_SOC20_vec_norand=MSW_SOC20_vec_norand,
        UKW_SIC07_vec_norand=UKW_SIC07_vec_norand,
        UKW_SOC20_vec_norand=UKW_SOC20_vec_norand)


# Pickle results to disk...

In [ ]:
results = dict(
    nation_results=nation_results, nation_rand_results=nation_rand_results,
    national_info=national_info,
    #X_pivot=X_pivot, GVA_pivot=GVA_pivot,  X_pivot_let=X_pivot_let, 
)

with open(f'../data/processed/MSW_2025.pickle', 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Or load already-computed results...

In [ ]:
with open(f'../data/processed/MSW_2025.pickle', 'rb') as handle:
    ZZ = pickle.load(handle)

locals().update(ZZ)

# Convert Notebook to HTML for sharing

In [ ]:
%%bash 
jupyter nbconvert --to html main_analysis.ipynb